# Use Case 1 — Agentic Spine, end to end

Build an agent, collect real rollouts, evaluate them, distill a student, and detect a self-heal failure — all on **one `Project`**, using a **real** local model (`Qwen/Qwen2.5-0.5B-Instruct`) instead of a mock. This is the "one trajectory schema across the lifecycle" idea behind AgentTune's agentic spine, shown end to end.

**What you'll see real, not mocked:**
- A real forward pass generating each rollout's completion (with real per-token logprobs)
- A real gradient step (forward + backward + optimizer.step()) during distillation
- A real `FailureDetector` catching a genuinely looping trajectory

In [1]:
import sys
sys.path.insert(0, ".")
from _real_backends import real_rollout_engine, real_sft_train_step

from agenttune.agentic.project import Project, agentic_metrics
from agenttune.agentic.events import EventLog, Event, EventKind

print("Ready.")


Ready.


## 1. Build + collect real rollouts

Ask the real model two real questions. Each rollout becomes a full-tier `EventLog` (it carries real per-token logprobs, not a placeholder).

In [2]:
p = Project()
engine = real_rollout_engine()

logs = p.collect_rollout(engine, [
    "What is the capital of France?",
    "What is 2+2?",
], max_steps=1)

for log in logs:
    completion = log.as_dataset_rows("sft")[0]["messages"][-1]["content"]
    print(f"tier={log.tier!r}  completion={completion!r}")

assert all(l.tier == "full" for l in logs)
assert p.native_trajectories[0].logprobs  # real per-token logprobs, not a stub
print(f"\n{len(p.native_trajectories)} native trajectories retained for RL/distillation use.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/workspace/agenttune/src/agenttune/agentic/project.py:140: UserWarning: No tools provided — running in single-turn mode.
  fn = create_rollout_fn(


tier='full'  completion='The capital of France is Paris.'
tier='full'  completion='The result of adding two identical numbers, 2 and 2, is simply:\n\n2 + 2 = 4\n\nThis is a basic arithmetic operation where you combine two equal quantities to find their total sum.'

2 native trajectories retained for RL/distillation use.


## 2. Evaluate — real programmatic metrics

No model call here: `agentic_metrics` computes tool-argument-correctness (TAC), tool-error-rate (TER), API-redundancy-ratio (ARR), etc. straight off the `EventLog`.

In [3]:
scores = [agentic_metrics(l) for l in p.trajectories]
for s in scores:
    print(s)


{'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0}
{'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0}


## 3. Distill — a real gradient step

`Project.distill` builds an SFT dataset from the collected trajectories and hands it to a trainer. Here the "trainer" actually runs one real forward+backward+`optimizer.step()` on a fresh copy of the model — a genuine (if tiny) training update, not a canned loss value.

In [4]:
class RealTrainer:
    last_dataset = None
    def __init__(self, *, train_dataset=None, **kw):
        RealTrainer.last_dataset = train_dataset
        self.train_dataset = train_dataset
    def train(self):
        return real_sft_train_step(self.train_dataset)

result = p.distill("Qwen2.5-0.5B-Instruct-student", trainer_factory=RealTrainer)
print(f"real train_loss = {result['train_loss']:.4f}  (n_rows={len(RealTrainer.last_dataset)})")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

real train_loss = 7.0329  (n_rows=2)


## 4. Heal — a real `FailureDetector` catching a real loop

Seed a trajectory where the same tool is called 4x in a row with the same result — a textbook `loop_collapse`. `Project.heal()` calls the *real* production `FailureDetector` (from `agenttune.decide.closed_loop`), not a mock.

In [5]:
loop_log = EventLog(tier="light")
for _ in range(4):
    loop_log.append(Event(EventKind.TOOL_CALL, {"action": {"name": "search", "arguments": {}}}))
    loop_log.append(Event(EventKind.TOOL_RESULT, {"output": "same"}))
p.add_trajectory(loop_log)

failures = p.heal(max_revisits=3)
for f in failures:
    print(f"failure_type={f.failure_type!r}  trajectory_id={f.trajectory_id!r}")

assert any(f.failure_type == "loop_collapse" for f in failures)


failure_type='loop_collapse'  trajectory_id='d85d2618-fcf1-4217-a909-d8b49982829e'


## 5. One Project, one lifecycle event stream

Every stage above emitted lifecycle events onto the *same* `Project` — this is the actual integration claim being demonstrated, not five disconnected demos.

In [6]:
stages = {ev.stage for ev in p.events()}
print("stages touched:", stages)
assert {"collect_rollout", "distill", "heal"} <= stages
print(f"trajectories={len(p.trajectories)}  native_trajectories={len(p.native_trajectories)}")


stages touched: {'heal', 'distill', 'collect_rollout'}
trajectories=3  native_trajectories=2


---
**See also:** notebook 3 for the related gap (`Project.heal()` finds the failure here, but does *not* gate/retrain automatically) and a real workaround.